In [ ]:
import json
import pprint

import requests

In [ ]:
OPENSEARCH_HOST = "https://hogehoge:10004"
OPENSEARCH_USER = "OpenSearchのユーザID"
OPENSEARCH_PASSWORD = "OpenSearchのパスワード"

### 登録する

##### agentを作成
##### https://docs.opensearch.org/latest/vector-search/ai-search/agentic-search/index/#step-4-create-an-agent
##### https://docs.opensearch.org/latest/vector-search/ai-search/agentic-search/agent-customization/#agent-interface-configuration
##### https://docs.opensearch.org/latest/vector-search/ai-search/agentic-search/flow-agent/

In [ ]:
LLM_OPENSEARCH_MODEL_ID = "claude 4.6 sonnetで作成したML ModelのID"  # claude 4.6 sonnetで作成したML Model

In [ ]:
# https://docs.opensearch.org/latest/ml-commons-plugin/agents-tools/tools/query-planning-tool/#register-parameters
agent_payload = {"name": "product-search-agent",
                 "type": "flow",
                 "description": "Agent for product search",
                 "tools": [{"type": "QueryPlanningTool",
                            "parameters": {"model_id": LLM_OPENSEARCH_MODEL_ID,
                                           "response_filter": "$.output.message.content[0].text",
                                           "query_planner_system_prompt": ("Return ONLY a valid OpenSearch search request body.\n"
                                                                           "Rules:\n"
                                                                           " - Return exactly one JSON object.\n"
                                                                           " - Do not use Markdown.\n"
                                                                           " - Do not use code fences.\n"
                                                                           " - Do not include explanations.\n"
                                                                           " - The response must be valid JSON.\n"
                                                                           " - Do not include '_source'.\n"
                                                                           " - Always set 'size' to 5 unless another value is explicitly requested.\n\n"
                                                                           "Search strategy:\n"
                                                                           " - Use a bool query.\n"
                                                                           " - Do NOT use hybrid queries.\n"
                                                                           " - Combine search clauses under bool.should.\n"
                                                                           " - Set minimum_should_match to 1.\n"
                                                                           " - Use bool.filter only for structured filters such as price, size, category, or brand.\n\n"
                                                                           "Semantic search:\n"
                                                                           " - When semantic search is beneficial, generate neural queries for:\n"
                                                                           "  - image_vector\n"
                                                                           "  - product_name_vector\n"
                                                                           "  - description_vector\n"
                                                                           "  - features_vector\n"
                                                                           " - Always include model_id, k, and boost.\n"
                                                                           " - Recommended values:\n"
                                                                           "  - image_vector: boost 4.0-6.0, k=15\n"
                                                                           "  - product_name_vector: boost 1.0-2.0, k=7\n"
                                                                           "  - description_vector: boost 1.0-2.0, k=7\n"
                                                                           "  - features_vector: boost 1.0-2.0, k=7\n"
                                                                           " - Adjust boost dynamically according to the user's intent.\n"
                                                                           " - Intent guidance:\n"
                                                                           "  - Prioritize image_vector for appearance, color, design, style, texture, pattern, silhouette, elegance, gorgeousness, cuteness, fashion, or atmosphere.\n"
                                                                           "  - Prioritize features_vector for materials, functions, specifications, comfort, washing instructions, hooks, pads, wires, and other product attributes.\n"
                                                                           "  - Prioritize product_name_vector when the product name is important.\n"
                                                                           "  - Prioritize description_vector for general semantic understanding.\n\n"
                                                                           "Lexical search:\n"
                                                                           " - When the query contains important keywords, product names, brands, categories, colors, materials, attributes, or tags, also generate BM25 queries using match or multi_match.\n"
                                                                           " - Combine BM25 queries with neural queries under bool.should.\n"
                                                                           " - Semantic search and BM25 should complement each other.")
                                          }
                           }]
                }

In [ ]:
agent_register_url = "{a}/_plugins/_ml/agents/_register".format(a=OPENSEARCH_HOST)
response = requests.post(url=agent_register_url,
                         auth=(OPENSEARCH_USER,
                               OPENSEARCH_PASSWORD),
                         headers={"Content-Type": "application/json"},
                         json=agent_payload,
                         verify=False
                        )
print(response.status_code)
print(response.json())

In [ ]:
agent_id = response.json()["agent_id"]
# agent_id = "作成したagentのID"

##### agentをテストする
##### https://docs.opensearch.org/latest/ml-commons-plugin/agents-tools/tools/query-planning-tool/#execute-parameters

In [ ]:
agent_test_url = "{a}/_plugins/_ml/agents/{b}/_execute".format(a=OPENSEARCH_HOST, b=agent_id)
# 03_dbindex_control.ipynbで作ったもの
target_index_name = "agentで検索を行う対象のDB indexの名前"
# 01_connector_model_control.ipynbで作ったもの
embedding_ml_model_id = "amazon nova multimodal embeddingで作成したML ModelのID"

In [ ]:
agent_test_payload = {"parameters":{"question": "10,000円程度でセクシーな商品を教えて",
                                    "index_name": target_index_name,
                                    "embedding_model_id": embedding_ml_model_id},
                      "dsl_query": True
                     }

In [ ]:
agent_test_response = requests.post(url=agent_test_url,
                                    auth=(OPENSEARCH_USER,
                                          OPENSEARCH_PASSWORD),
                                    headers={"Content-Type": "application/json"},
                                    json=agent_test_payload,
                                    verify=False
                                   )
print(agent_test_response.status_code)
pprint.pprint(agent_test_response.json())

##### pipelineを作成
##### https://docs.opensearch.org/latest/vector-search/ai-search/agentic-search/neural-search/?utm_source=chatgpt.com#step-2c-create-a-search-pipeline
##### https://docs.opensearch.org/latest/search-plugins/search-pipelines/agentic-context-processor/#example

In [ ]:
EMBEDDING_OPENSEARCH_MODEL_ID = "amazon nova multimodal embeddingで作成したML ModelのID"  # amazon nova multimodal embeddingで作成したML Model

In [ ]:
pipeline_payload = {"description": "product agentic search pipeline",
                    "request_processors": [{"agentic_query_translator": {"agent_id": agent_id,
                                                                         "embedding_model_id": EMBEDDING_OPENSEARCH_MODEL_ID}
                                           }],
                    "response_processors": [{"agentic_context": {"dsl_query": True,
                                                                 "agent_steps_summary": True}
                                            }]
                   }

In [ ]:
pipeline_name = "my-pipeline-name"  # 作成するpipelineに付ける名前
pipeline_register_url = "{a}/_search/pipeline/{b}".format(a=OPENSEARCH_HOST, b=pipeline_name)
response_2 = requests.put(url=pipeline_register_url,
                          auth=(OPENSEARCH_USER,
                                OPENSEARCH_PASSWORD),
                          headers={"Content-Type": "application/json"},
                          json=pipeline_payload,
                          verify=False
                         )
print(response_2.status_code)
print(response_2.json())

##### 作成済みのagentやpipelineを確認する場合

In [ ]:
# https://docs.opensearch.org/latest/ml-commons-plugin/api/agent-apis/search-agent
agent_check_url = "{a}/_plugins/_ml/agents/_search".format(a=OPENSEARCH_HOST)
agent_check_payload = {"query": {"match_all": {}},
                       "size": 1000}
response_3 = requests.post(url=agent_check_url,
                           auth=(OPENSEARCH_USER,
                                 OPENSEARCH_PASSWORD),
                           headers={"Content-Type": "application/json"},
                           json=agent_check_payload,
                           verify=False)
print(response_3.status_code)
pprint.pprint(response_3.json())

In [ ]:
# https://docs.opensearch.org/latest/search-plugins/search-pipelines/retrieving-search-pipeline/
pipeline_check_url = "{a}/_search/pipeline".format(a=OPENSEARCH_HOST)
response_4 = requests.get(url=pipeline_check_url,
                          auth=(OPENSEARCH_USER,
                                OPENSEARCH_PASSWORD),
                          headers={"Content-Type": "application/json"},
                          verify=False)
print(response_4.status_code)
pprint.pprint(response_4.json())

### 削除する

##### pipelineを削除
##### https://docs.opensearch.org/latest/search-plugins/search-pipelines/deleting-search-pipeline/

In [ ]:
delete_pipeline_name = "my-pipeline-name"  # 作成するpipelineの名前

In [ ]:
pipeline_delete_url = "{a}/_search/pipeline/{b}".format(a=OPENSEARCH_HOST, b=delete_pipeline_name)
response_5 = requests.delete(pipeline_delete_url,
                             auth=(OPENSEARCH_USER,
                                   OPENSEARCH_PASSWORD),
                             verify=False
                            )
print(response_5.status_code)
print(response_5.json())

##### agentを削除
##### https://docs.opensearch.org/latest/ml-commons-plugin/api/agent-apis/delete-agent/

In [ ]:
delete_agent_id = "削除するagentのID"

In [ ]:
agent_delete_url = "{a}/_plugins/_ml/agents/{b}".format(a=OPENSEARCH_HOST, b=delete_agent_id)
response_6 = requests.delete(agent_delete_url,
                             auth=(OPENSEARCH_USER,
                                   OPENSEARCH_PASSWORD),
                             verify=False
                            )
print(response_6.status_code)
print(response_6.json())